In [ ]:
!rm -rf ~/.cache/huggingface/datasets
!rm -rf ~/.cache/huggingface/modules
!rm -rf ~/.cache/huggingface/hub


In [ ]:
!pip install --upgrade datasets fsspec huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [ ]:
# multinews
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import load_dataset

multi_news = load_dataset("multi_news")
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def summarize(text, max_input_length=512, max_summary_length=50):
    input_text = "summarize: " + text
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=max_input_length, padding="max_length")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_summary_length,
        num_beams=4,
        early_stopping=True
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

for i in range(5):
    doc = multi_news["test"][i]["document"]
    print(f"\nDocument {i+1}:\n", doc[:500], "...\n")
    summary = summarize(doc)
    print(f"Generated Summary {i+1}:\n", summary)


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Document 1:
 GOP Eyes Gains As Voters In 11 States Pick Governors 
 
 Enlarge this image toggle caption Jim Cole/AP Jim Cole/AP 
 
 Voters in 11 states will pick their governors tonight, and Republicans appear on track to increase their numbers by at least one, with the potential to extend their hold to more than two-thirds of the nation's top state offices. 
 
 Eight of the gubernatorial seats up for grabs are now held by Democrats; three are in Republican hands. Republicans currently hold 29 governorships, ...

Generated Summary 1:
 eight of the gubernatorial seats up for grabs are now held by Democrats. only three of tonight's contests are considered competitive, all in states where incumbent Democratic governors aren't running again.

Document 2:
 
 
 
 
 UPDATE: 4/19/2001 Read Richard Metzger: How I, a married, middle-aged man, became an accidental spokesperson for gay rights overnight on Boing Boing 
 
 It’s time to clarify a few details about the controversial “Hey Facebook wha

In [ ]:
!pip install fsspec==2023.6.0 --quiet


In [ ]:
!pip install -U "fsspec==2023.6.0" "huggingface_hub>=0.30.0" --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.1/512.1 kB 8.0 MB/s eta 0:00:00


In [31]:
#CNN
rom datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

def safe_load_dataset():
    """Handles all possible dataset loading scenarios"""
    try:
        # Try CNN/DailyMail first
        return load_dataset("cnn_dailymail", name="3.0.0", trust_remote_code=True)
    except Exception as e:
        print(f"CNN/DailyMail loading failed: {e}")
        try:
            # Try SAMSum fallback
            return load_dataset("samsum", trust_remote_code=True)
        except:
            # Ultimate fallback - use local dummy data
            print("Using local dummy data")
            return {
                "test": [{
                    "article": "The quick brown fox jumps over the lazy dog. " * 20,
                    "highlights": "Fox jumps over dog."
                }]
            }

cnn_dm = safe_load_dataset()
dataset = cnn_dm["test"]

tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

sample = dataset[0]["article"]
inputs = tokenizer("summarize: " + sample,
                  return_tensors="pt",
                  truncation=True,
                  max_length=512).to(device)

summary_ids = model.generate(inputs["input_ids"],
                           max_length=150,
                           num_beams=4,
                           early_stopping=True)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\n📜 Original (truncated):")
print(sample[:200] + "...")
print("\n📝 Generated Summary:")
print(summary)

CNN/DailyMail loading failed: Invalid pattern: '**' can only be an entire path component
Using local dummy data

📜 Original (truncated):
The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog. The quick brown fox jumps over the lazy dog. The quick brown fox ...

📝 Generated Summary:
the quick brown fox jumps over the lazy dog. the quick brown fox jumps over the lazy dog.


In [24]:

#wikisum
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration

dataset = load_dataset("d0rj/wikisum")
example = dataset["train"][0]
print("Available fields in sample:", example.keys())
input_text = "summarize: " + example["article"]
reference_summary = example["summary"]
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512, padding="max_length")
inputs = {k: v.to(device) for k, v in inputs.items()}
summary_ids = model.generate(inputs["input_ids"], max_length=150, num_beams=4, early_stopping=True)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("\nGenerated Summary:\n", summary)
print("\nReference Summary:\n", reference_summary)


Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/35775 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

NotImplementedError: Loading a dataset cached in a LocalFileSystem is not supported.

In [14]:
#multinews
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import load_dataset
import evaluate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model_name = "t5-small"
print("\nLoading T5 model and tokenizer...")
try:
    tokenizer = T5Tokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Failed to load model: {e}")
    exit(1)

print("\nLoading MultiNews dataset...")
try:
    multi_news = load_dataset(
        "multi_news",
        cache_dir="/tmp/hf_cache",
        download_mode="force_redownload"
    )
    print(f"Total Test Samples: {len(multi_news['test'])}")
    eval_data = multi_news["test"].select(range(min(5, len(multi_news["test"]))))
    print(f"Using {len(eval_data)} samples for evaluation")
except Exception as e:
    print(f"❌ Failed to load dataset: {e}")
    eval_data = []

rouge = evaluate.load("rouge")
predictions = []
references = []

print("\n" + "="*50)
print("Generating Summaries")
print("="*50)

if not eval_data:
    print("⚠️ Skipping summary generation due to dataset loading failure.")
else:
    for i, sample in enumerate(eval_data):
        try:
            doc = sample["document"]
            ref = sample["summary"]
            input_text = "summarize: " + doc
            inputs = tokenizer(
                input_text,
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding="max_length"
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model.generate(
                    input_ids=inputs["input_ids"],
                    max_length=128,
                    num_beams=4,
                    early_stopping=True
                )

            pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
            predictions.append(pred)
            references.append(ref)

            print(f"\nExample {i+1}:")
            print(f"Generated: {pred}")
            print(f"Reference: {ref}")
            print("-" * 50)

        except Exception as e:
            print(f" Error processing sample {i}: {e}")

if predictions and references:
    results = rouge.compute(predictions=predictions, references=references)
    print("\nEvaluation Results:")
    print(f"ROUGE-1: {results['rouge1']:.4f}")
    print(f"ROUGE-2: {results['rouge2']:.4f}")
    print(f"ROUGE-L: {results['rougeL']:.4f}")

del model
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("\nEvaluation completed!")


Using device: cpu

Loading T5 model and tokenizer...
Model loaded successfully!

Loading MultiNews dataset...


Generating train split:   0%|          | 0/44972 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5622 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5622 [00:00<?, ? examples/s]

❌ Failed to load dataset: Loading a dataset cached in a LocalFileSystem is not supported.

Generating Summaries
⚠️ Skipping summary generation due to dataset loading failure.

Evaluation completed!


In [16]:
#cnn
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model_name = "t5-small"
print("\nLoading T5 model and tokenizer...")
try:
    tokenizer = T5Tokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    exit(1)

print("\nLoading CNN/DailyMail dataset...")
try:
    cnn_dm = load_dataset("cnn_dailymail", "3.0.0")
    dataset = cnn_dm["test"]
    eval_data = dataset.select(range(min(5, len(dataset))))
    print(f"✅ Dataset loaded with {len(eval_data)} samples for evaluation")
except Exception as e:
    print(f"❌ Failed to load dataset: {e}")
    exit(1)

rouge = evaluate.load("rouge")
predictions = []
references = []

print("\n" + "="*50)
print("Generating Summaries")
print("="*50)

for i, sample in enumerate(eval_data):
    try:
        article = sample["article"]
        ref = sample["highlights"]

        input_text = "summarize: " + article
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding="max_length"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            output_ids = model.generate(
                inputs["input_ids"],
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

        pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        predictions.append(pred)
        references.append(ref)

        print(f"\nExample {i+1}:")
        print(f"Generated: {pred}")
        print(f"Reference: {ref}")
        print("-" * 50)

    except Exception as e:
        print(f"❌ Error processing sample {i}: {e}")

if predictions and references:
    results = rouge.compute(predictions=predictions, references=references)
    print("\nEvaluation Results:")
    print(f"ROUGE-1: {results['rouge1']:.4f}")
    print(f"ROUGE-2: {results['rouge2']:.4f}")
    print(f"ROUGE-L: {results['rougeL']:.4f}")

del model
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("\n✅ Evaluation completed!")


Using device: cpu

Loading T5 model and tokenizer...
✅ Model loaded successfully!

Loading CNN/DailyMail dataset...
❌ Failed to load dataset: Invalid pattern: '**' can only be an entire path component

Generating Summaries

✅ Evaluation completed!
